# 02 - Enriquecimiento y unificacion

Objetivos de este notebook:
- Integrar Taxi Zones para origen (PU) y destino (DO).
- Unificar viajes de Yellow y Green en una estructura comun.
- Normalizar catalogos operativos (vendor, rate code, payment type, trip type, store and forward).
- Publicar tablas curadas en Snowflake.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from urllib.request import Request, urlopen
import datetime
import glob
import os

In [2]:
def log_step(message):
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def build_spark(app_name):
    jars_dir = os.environ.get('SPARK_JARS_DIR', '/home/jovyan/jars')
    jar_paths = sorted(glob.glob(os.path.join(jars_dir, '*.jar')))
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
    )
    if jar_paths:
        jars_csv = ','.join(jar_paths)
        classpath = ':'.join(jar_paths)
        log_step(f'Loading JARs: {jar_paths}')
        builder = (
            builder
            .config('spark.jars', jars_csv)
            .config('spark.driver.extraClassPath', classpath)
            .config('spark.executor.extraClassPath', classpath)
        )
    else:
        log_step(f'No JARs found in {jars_dir}')
    return builder.getOrCreate()


def assert_snowflake_connector(spark):
    class_names = ['net.snowflake.spark.snowflake.DefaultSource', 'snowflake.DefaultSource']
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found. Run `bash scripts/download_snowflake_jars.sh` from the repo root, restart the container, and rerun this notebook.')


app = build_spark('02_enriquecimiento_y_unificacion')
assert_snowflake_connector(app)

# Credenciales Snowflake desde variables de ambiente
curated_schema = os.environ.get('SF_CURATED_SCHEMA', 'CURATED')

sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

sf_utils = app._jvm.net.snowflake.spark.snowflake.Utils

def sf_run(sql):
    jsf = app._jvm.java.util.HashMap()
    for k, v in sf_option.items():
        jsf.put(k, v)
    sf_utils.runQuery(jsf, sql)

# El schema CURATED debe existir antes de cualquier write Spark. No basta usar preactions,
# porque el conector puede validar el destino antes de ejecutar esas acciones.
sf_run(f'CREATE SCHEMA IF NOT EXISTS {curated_schema}')

def write_snowflake_table(df, dbtable, mode='overwrite', preactions=None):
    writer = (
        df.write.format('snowflake')
        .options(**sf_option)
        .option('dbtable', dbtable)
        .mode(mode)
    )
    if preactions:
        writer = writer.option('preactions', preactions)
    writer.save()

log_step(f'Notebook 02 ready: curated_schema={curated_schema}')

[2026-04-07 22:32:11Z] Loading JARs: ['/home/jovyan/jars/snowflake-jdbc-3.28.0.jar', '/home/jovyan/jars/spark-snowflake_2.12-3.1.8.jar']
[2026-04-07 22:32:13Z] Snowflake connector detected: net.snowflake.spark.snowflake.DefaultSource
[2026-04-07 22:32:18Z] Notebook 02 ready: curated_schema=CURATED


In [3]:
# Lectura de tablas RAW generadas en el notebook 01
def normalize_columns(df):
    return df.toDF(*[c.lower() for c in df.columns])

df_yellow_raw = normalize_columns(
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.YELLOW_TRIPS')
    .load()
)

df_green_raw = normalize_columns(
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.GREEN_TRIPS')
    .load()
)

yellow_rows = df_yellow_raw.count()
green_rows = df_green_raw.count()
log_step(f'Loaded RAW tables: yellow_rows={yellow_rows}, green_rows={green_rows}')

[2026-04-07 22:32:38Z] Loaded RAW tables: yellow_rows=0, green_rows=446742


In [4]:
# Estandarizacion de timestamps y claves operativas para unificar tipos de taxi
def base_columns(df):
    columns = set(df.columns)

    def existing_col(*names):
        for name in names:
            if name in columns:
                return F.col(name)
        return F.lit(None)

    pickup_col = existing_col('tpep_pickup_datetime', 'lpep_pickup_datetime', 'pickup_datetime')
    dropoff_col = existing_col('tpep_dropoff_datetime', 'lpep_dropoff_datetime', 'dropoff_datetime')
    vendor_col = existing_col('vendorid', 'vendor_id').cast('int')
    rate_col = existing_col('ratecodeid', 'rate_code', 'rate_code_id').cast('int')
    payment_col = existing_col('payment_type', 'payment_type_code').cast('int')
    trip_type_col = existing_col('trip_type', 'trip_type_code').cast('int')
    store_and_fwd_col = existing_col('store_and_fwd_flag', 'store_and_fwd_flag_norm')

    return (
        df
        .withColumn('pickup_datetime_utc', pickup_col.cast('timestamp'))
        .withColumn('dropoff_datetime_utc', dropoff_col.cast('timestamp'))
        .withColumn('vendor_id', vendor_col)
        .withColumn('rate_code_id', rate_col)
        .withColumn('payment_type_code', payment_col)
        .withColumn('trip_type_code', trip_type_col)
        .withColumn('store_and_fwd_flag_norm', F.upper(F.trim(store_and_fwd_col.cast('string'))))
    )

In [5]:
# Se aplica el mismo esquema de negocio para yellow y green
df_yellow = base_columns(df_yellow_raw).withColumn('taxi_type', F.lit('yellow'))
df_green = base_columns(df_green_raw).withColumn('taxi_type', F.lit('green'))

unified_cols = [
    'trip_nk', 'run_id', 'service_type', 'taxi_type',
    'source_year', 'source_month', 'ingested_at_utc', 'source_path',
    'pickup_datetime_utc', 'dropoff_datetime_utc',
    'pulocationid', 'dolocationid',
    'vendor_id', 'rate_code_id', 'payment_type_code', 'trip_type_code', 'store_and_fwd_flag_norm',
    'passenger_count', 'trip_distance',
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
    'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount'
]

for c in unified_cols:
    if c not in df_yellow.columns:
        df_yellow = df_yellow.withColumn(c, F.lit(None))
    if c not in df_green.columns:
        df_green = df_green.withColumn(c, F.lit(None))

df_trips_unified = df_yellow.select(*unified_cols).unionByName(df_green.select(*unified_cols))

In [6]:
# Integracion Taxi Zones desde lookup oficial TLC / NYC Open Data
def download_reference_file(url_candidates, filename):
    local_dir = os.environ.get('LOCAL_REFERENCE_DIR', '/tmp/nyc_taxi_reference')
    os.makedirs(local_dir, exist_ok=True)
    local_path = os.path.join(local_dir, filename)

    if os.path.exists(local_path):
        log_step(f'Using cached reference file: {local_path}')
        return local_path

    errors = []
    for url in url_candidates:
        if not url:
            continue
        try:
            log_step(f'Downloading reference file: {url}')
            request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(request) as response, open(local_path, 'wb') as output_file:
                output_file.write(response.read())
            return local_path
        except Exception as exc:
            errors.append(f'{url} -> {str(exc)[:160]}')
            log_step(f'Reference download failed: {url} | {str(exc)[:160]}')

    raise RuntimeError('Could not download Taxi Zone lookup from any configured source. Tried: ' + ' || '.join(errors))

zones_url_candidates = [
    os.environ.get('TAXI_ZONE_PATH', ''),
    'https://s3.amazonaws.com/nyc-tlc/misc/taxi+_zone_lookup.csv',
    'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv',
    'https://data.cityofnewyork.us/api/views/8meu-9t5y/rows.csv?accessType=DOWNLOAD',
]
zones_path = download_reference_file(zones_url_candidates, 'taxi_zone_lookup.csv')

df_zones_raw = app.read.option('header', True).option('inferSchema', True).csv(zones_path)

# Handle different column naming conventions across CSV sources
cols = df_zones_raw.columns
loc_col = 'LocationID' if 'LocationID' in cols else 'Location ID' if 'Location ID' in cols else cols[0]
borough_col = 'Borough' if 'Borough' in cols else 'borough'
zone_col = 'Zone' if 'Zone' in cols else 'zone'
svc_col = 'service_zone' if 'service_zone' in cols else None

log_step(f'Taxi zones columns: {cols}, using loc={loc_col}, borough={borough_col}, zone={zone_col}, svc={svc_col}')

df_zones = df_zones_raw.withColumn('locationid', F.col(f'`{loc_col}`').cast('int'))
df_zones = df_zones.withColumn('borough', F.trim(F.col(f'`{borough_col}`')))
df_zones = df_zones.withColumn('zone', F.trim(F.col(f'`{zone_col}`')))
if svc_col:
    df_zones = df_zones.withColumn('service_zone', F.trim(F.col(svc_col)))
else:
    df_zones = df_zones.withColumn('service_zone', F.lit('N/A'))
df_zones = df_zones.dropDuplicates(['locationid'])

df_zones_pu = (
    df_zones
    .select(
        F.col('locationid').alias('pu_location_id'),
        F.col('borough').alias('pu_borough'),
        F.col('zone').alias('pu_zone'),
        F.col('service_zone').alias('pu_service_zone')
    )
)

df_zones_do = (
    df_zones
    .select(
        F.col('locationid').alias('do_location_id'),
        F.col('borough').alias('do_borough'),
        F.col('zone').alias('do_zone'),
        F.col('service_zone').alias('do_service_zone')
    )
)

[2026-04-07 22:32:39Z] Downloading reference file: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
[2026-04-07 22:32:40Z] Taxi zones columns: ['LocationID', 'Borough', 'Zone', 'service_zone'], using loc=LocationID, borough=Borough, zone=Zone, svc=service_zone


In [7]:
# Normalizacion de catalogos (codigos a descripciones)
catalog_vendor = [
    (1, 'Creative Mobile Technologies'),
    (2, 'VeriFone Inc.'),
    (6, 'Myle Technologies Inc.')
]

catalog_ratecode = [
    (1, 'Standard rate'),
    (2, 'JFK'),
    (3, 'Newark'),
    (4, 'Nassau or Westchester'),
    (5, 'Negotiated fare'),
    (6, 'Group ride')
]

catalog_payment_type = [
    (1, 'Credit card'),
    (2, 'Cash'),
    (3, 'No charge'),
    (4, 'Dispute'),
    (5, 'Unknown'),
    (6, 'Voided trip')
]

catalog_trip_type = [
    (1, 'Street-hail'),
    (2, 'Dispatch')
]

catalog_store_and_fwd = [
    ('Y', 'Stored and forwarded'),
    ('N', 'Not a stored trip')
]

df_cat_vendor = app.createDataFrame(catalog_vendor, ['vendor_id', 'vendor_desc'])
df_cat_rate = app.createDataFrame(catalog_ratecode, ['rate_code_id', 'rate_code_desc'])
df_cat_payment = app.createDataFrame(catalog_payment_type, ['payment_type_code', 'payment_type_desc'])
df_cat_trip = app.createDataFrame(catalog_trip_type, ['trip_type_code', 'trip_type_desc'])
df_cat_saf = app.createDataFrame(catalog_store_and_fwd, ['store_and_fwd_flag_norm', 'store_and_fwd_desc'])

In [8]:
df_trips_enriched = (
    df_trips_unified
    .withColumn('pulocationid', F.col('pulocationid').cast('int'))
    .withColumn('dolocationid', F.col('dolocationid').cast('int'))
    .join(df_zones_pu, F.col('pulocationid') == F.col('pu_location_id'), 'left')
    .join(df_zones_do, F.col('dolocationid') == F.col('do_location_id'), 'left')
    .join(df_cat_vendor, on='vendor_id', how='left')
    .join(df_cat_rate, on='rate_code_id', how='left')
    .join(df_cat_payment, on='payment_type_code', how='left')
    .join(df_cat_trip, on='trip_type_code', how='left')
    .join(df_cat_saf, on='store_and_fwd_flag_norm', how='left')
    .withColumn('trip_duration_minutes', (F.unix_timestamp('dropoff_datetime_utc') - F.unix_timestamp('pickup_datetime_utc')) / 60.0)
    .withColumn('trip_date', F.to_date('pickup_datetime_utc'))
)

In [9]:
# Publicacion de tablas curadas (dimensiones pequeñas via Spark, tabla grande via SQL en Snowflake)
sf_run(f'CREATE SCHEMA IF NOT EXISTS {curated_schema}')

write_snowflake_table(df_zones, f'{curated_schema}.DIM_TAXI_ZONES')
write_snowflake_table(df_cat_vendor, f'{curated_schema}.DIM_VENDOR')
write_snowflake_table(df_cat_rate, f'{curated_schema}.DIM_RATE_CODE')
write_snowflake_table(df_cat_payment, f'{curated_schema}.DIM_PAYMENT_TYPE')
write_snowflake_table(df_cat_trip, f'{curated_schema}.DIM_TRIP_TYPE')
write_snowflake_table(df_cat_saf, f'{curated_schema}.DIM_STORE_AND_FWD')

log_step('Dimension tables written. Now writing FCT_TRIPS_ENRICHED via Snowflake SQL (faster)...')

# Use Snowflake SQL to build FCT_TRIPS_ENRICHED directly — avoids Spark memory issues
sf_run(f'DROP TABLE IF EXISTS {curated_schema}.FCT_TRIPS_ENRICHED')

# Build unified enriched table directly in Snowflake using SQL
# Use IFNULL/TRY_CAST for columns that may not exist in all tables
sf_run(f"""
CREATE TABLE {curated_schema}.FCT_TRIPS_ENRICHED AS
SELECT
    y.TRIP_NK, y.RUN_ID, y.SERVICE_TYPE, 'yellow' AS TAXI_TYPE,
    y.SOURCE_YEAR, y.SOURCE_MONTH, y.INGESTED_AT_UTC, y.SOURCE_PATH,
    y.TPEP_PICKUP_DATETIME AS PICKUP_DATETIME_UTC,
    y.TPEP_DROPOFF_DATETIME AS DROPOFF_DATETIME_UTC,
    y.PULOCATIONID, y.DOLOCATIONID,
    TRY_CAST(y.VENDORID AS INT) AS VENDOR_ID,
    TRY_CAST(y.RATECODEID AS INT) AS RATE_CODE_ID,
    TRY_CAST(y.PAYMENT_TYPE AS INT) AS PAYMENT_TYPE_CODE,
    NULL::INT AS TRIP_TYPE_CODE,
    UPPER(TRIM(y.STORE_AND_FWD_FLAG::VARCHAR)) AS STORE_AND_FWD_FLAG_NORM,
    y.PASSENGER_COUNT, y.TRIP_DISTANCE,
    y.FARE_AMOUNT, y.EXTRA, y.MTA_TAX, y.TIP_AMOUNT, y.TOLLS_AMOUNT,
    y.IMPROVEMENT_SURCHARGE, y.CONGESTION_SURCHARGE,
    TRY_CAST(y.AIRPORT_FEE AS FLOAT) AS AIRPORT_FEE,
    y.TOTAL_AMOUNT,
    TIMESTAMPDIFF(SECOND, y.TPEP_PICKUP_DATETIME, y.TPEP_DROPOFF_DATETIME) / 60.0 AS TRIP_DURATION_MINUTES,
    TO_DATE(y.TPEP_PICKUP_DATETIME) AS TRIP_DATE,
    pz.BOROUGH AS PU_BOROUGH, pz.ZONE AS PU_ZONE, pz.SERVICE_ZONE AS PU_SERVICE_ZONE,
    dz.BOROUGH AS DO_BOROUGH, dz.ZONE AS DO_ZONE, dz.SERVICE_ZONE AS DO_SERVICE_ZONE,
    v.VENDOR_DESC,
    r.RATE_CODE_DESC,
    p.PAYMENT_TYPE_DESC,
    NULL::VARCHAR AS TRIP_TYPE_DESC,
    s.STORE_AND_FWD_DESC
FROM RAW.YELLOW_TRIPS y
LEFT JOIN {curated_schema}.DIM_TAXI_ZONES pz ON y.PULOCATIONID = pz.LOCATIONID
LEFT JOIN {curated_schema}.DIM_TAXI_ZONES dz ON y.DOLOCATIONID = dz.LOCATIONID
LEFT JOIN {curated_schema}.DIM_VENDOR v ON TRY_CAST(y.VENDORID AS INT) = v.VENDOR_ID
LEFT JOIN {curated_schema}.DIM_RATE_CODE r ON TRY_CAST(y.RATECODEID AS INT) = r.RATE_CODE_ID
LEFT JOIN {curated_schema}.DIM_PAYMENT_TYPE p ON TRY_CAST(y.PAYMENT_TYPE AS INT) = p.PAYMENT_TYPE_CODE
LEFT JOIN {curated_schema}.DIM_STORE_AND_FWD s ON UPPER(TRIM(y.STORE_AND_FWD_FLAG::VARCHAR)) = s.STORE_AND_FWD_FLAG_NORM

UNION ALL

SELECT
    g.TRIP_NK, g.RUN_ID, g.SERVICE_TYPE, 'green' AS TAXI_TYPE,
    g.SOURCE_YEAR, g.SOURCE_MONTH, g.INGESTED_AT_UTC, g.SOURCE_PATH,
    g.LPEP_PICKUP_DATETIME AS PICKUP_DATETIME_UTC,
    g.LPEP_DROPOFF_DATETIME AS DROPOFF_DATETIME_UTC,
    g.PULOCATIONID, g.DOLOCATIONID,
    TRY_CAST(g.VENDORID AS INT) AS VENDOR_ID,
    TRY_CAST(g.RATECODEID AS INT) AS RATE_CODE_ID,
    TRY_CAST(g.PAYMENT_TYPE AS INT) AS PAYMENT_TYPE_CODE,
    TRY_CAST(g.TRIP_TYPE AS INT) AS TRIP_TYPE_CODE,
    UPPER(TRIM(g.STORE_AND_FWD_FLAG::VARCHAR)) AS STORE_AND_FWD_FLAG_NORM,
    g.PASSENGER_COUNT, g.TRIP_DISTANCE,
    g.FARE_AMOUNT, g.EXTRA, g.MTA_TAX, g.TIP_AMOUNT, g.TOLLS_AMOUNT,
    g.IMPROVEMENT_SURCHARGE, g.CONGESTION_SURCHARGE,
    NULL::FLOAT AS AIRPORT_FEE,
    g.TOTAL_AMOUNT,
    TIMESTAMPDIFF(SECOND, g.LPEP_PICKUP_DATETIME, g.LPEP_DROPOFF_DATETIME) / 60.0 AS TRIP_DURATION_MINUTES,
    TO_DATE(g.LPEP_PICKUP_DATETIME) AS TRIP_DATE,
    pz.BOROUGH AS PU_BOROUGH, pz.ZONE AS PU_ZONE, pz.SERVICE_ZONE AS PU_SERVICE_ZONE,
    dz.BOROUGH AS DO_BOROUGH, dz.ZONE AS DO_ZONE, dz.SERVICE_ZONE AS DO_SERVICE_ZONE,
    v.VENDOR_DESC,
    r.RATE_CODE_DESC,
    p.PAYMENT_TYPE_DESC,
    t.TRIP_TYPE_DESC,
    s.STORE_AND_FWD_DESC
FROM RAW.GREEN_TRIPS g
LEFT JOIN {curated_schema}.DIM_TAXI_ZONES pz ON g.PULOCATIONID = pz.LOCATIONID
LEFT JOIN {curated_schema}.DIM_TAXI_ZONES dz ON g.DOLOCATIONID = dz.LOCATIONID
LEFT JOIN {curated_schema}.DIM_VENDOR v ON TRY_CAST(g.VENDORID AS INT) = v.VENDOR_ID
LEFT JOIN {curated_schema}.DIM_RATE_CODE r ON TRY_CAST(g.RATECODEID AS INT) = r.RATE_CODE_ID
LEFT JOIN {curated_schema}.DIM_PAYMENT_TYPE p ON TRY_CAST(g.PAYMENT_TYPE AS INT) = p.PAYMENT_TYPE_CODE
LEFT JOIN {curated_schema}.DIM_TRIP_TYPE t ON TRY_CAST(g.TRIP_TYPE AS INT) = t.TRIP_TYPE_CODE
LEFT JOIN {curated_schema}.DIM_STORE_AND_FWD s ON UPPER(TRIM(g.STORE_AND_FWD_FLAG::VARCHAR)) = s.STORE_AND_FWD_FLAG_NORM
""")

log_step('FCT_TRIPS_ENRICHED created via Snowflake SQL')

[2026-04-07 22:33:09Z] Dimension tables written. Now writing FCT_TRIPS_ENRICHED via Snowflake SQL (faster)...
[2026-04-07 22:33:14Z] FCT_TRIPS_ENRICHED created via Snowflake SQL


In [10]:
# Validaciones rapidas
df_trips_enriched.groupBy('taxi_type').count().orderBy('taxi_type').show()

df_trips_enriched.select(
    'trip_nk', 'taxi_type',
    'pickup_datetime_utc', 'dropoff_datetime_utc',
    'pulocationid', 'pu_borough', 'pu_zone',
    'dolocationid', 'do_borough', 'do_zone',
    'vendor_id', 'vendor_desc',
    'rate_code_id', 'rate_code_desc',
    'payment_type_code', 'payment_type_desc',
    'trip_type_code', 'trip_type_desc',
    'store_and_fwd_flag_norm', 'store_and_fwd_desc',
    'total_amount', 'trip_duration_minutes'
).show(20, truncate=False)

+---------+------+
|taxi_type| count|
+---------+------+
|    green|446742|
+---------+------+

+----------------------------------------------------------------+---------+-------------------+--------------------+------------+----------+---------------------------+------------+----------+-----------------------+---------+----------------------------+------------+--------------+-----------------+-----------------+--------------+--------------+-----------------------+------------------+------------+---------------------+
|trip_nk                                                         |taxi_type|pickup_datetime_utc|dropoff_datetime_utc|pulocationid|pu_borough|pu_zone                    |dolocationid|do_borough|do_zone                |vendor_id|vendor_desc                 |rate_code_id|rate_code_desc|payment_type_code|payment_type_desc|trip_type_code|trip_type_desc|store_and_fwd_flag_norm|store_and_fwd_desc|total_amount|trip_duration_minutes|
+---------------------------------------------